***

# **BLS Queries**

***

In this file, we want to try and pull the total number of employment from BLS for different industries. To give some context, the API works by specifying a specific series id, which can be used to pull data from a specific table from BLS. It can also be used to select information from a table, parameters like specific counties or specific industry can be referenced. The survey we are trying to pull from is State and Employment, Hours, and Earnings; which can be found in the following link: https://www.bls.gov/help/hlpforma.htm#EW. 

***

## **Prepare Workspace**

***

In [ ]:
# Packages
import pandas as pd
import json
import requests
import os
from functools import reduce
from tqdm import tqdm

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    
    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'BLS Data')
    path_main = os.path.join(path_sp, 'Data')
    
if user in ['jchoy', 'aazawii']:
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')

path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'BLS', 'config')



print(user)
print(path_git)

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

# Base URL for API V2
url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

# Set API key
exec(open(os.path.join(path_config, 'api_key.txt')).read())
key = dict_api[user]
key = '?registrationkey={}'.format(key)

***

## **Pull Data**

***

Check the configuration file for importing data

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'BLS Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name'    ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'          ]['Input'].values[0]
survey             = df_params[df_params['Type'] == 'sample'            ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'         ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'        ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'       ]['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'          ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'        ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'          ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'      ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'         ]['Input'].values[0]
MOE = 'Yes'

# view
print(indicator_name)
print(estimate)
print(survey)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

In [ ]:
# Compiling all of the necessary steps into one chunk to test.

# Reading in MSA inputs
df_peer_msa = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'MSA', dtype = {'msa': object})
# df_peer_msa = df_params[['msa', 'msa_label']]

# State codes
df_states = pd.read_excel(os.path.join(path_config0, "Area Codes.xlsx"), sheet_name = 'MSAcodes', dtype = {'State FIPS': object, 'MSA_ID': object})
df_peer_msa = df_peer_msa.merge(df_states[['MSA_ID', 'State FIPS']], left_on = 'msa', right_on = 'MSA_ID', how = 'left')
df_peer_msa = df_peer_msa.drop('MSA_ID', axis = 1)
df_peer_msa = df_peer_msa.dropna()

# Industries and data types
df_industries = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                              , sheet_name = 'industry_codes'
                              , dtype = {'industry_code': object})
df_industries = df_industries[df_industries['Include'] == 'Yes']
df_industries = df_industries[df_industries['Indicator Name'].str.contains(indicator_name)]
df_datatypes = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                             , sheet_name = 'datatype_codes'
                              , dtype = {'data_type_code': object})
df_datatypes = df_datatypes[df_datatypes['Include'] == 'Yes']
df_datatypes = df_datatypes[df_datatypes['Indicator Name'].str.contains(indicator_name)]

# Inputs
sectors    = list(df_industries['industry_code'].values)
data_type  = df_datatypes['data_type_code'].values[0]

# View
print(sectors   )
print(data_type )
df_peer_msa.head()

Pull data

In [ ]:
# Query BLS data based on parameters set above
# Convert total jobs to raw counts (instead of per thousand jobs)
# Reshape data 

print("Begin process pulling BLS data")
print('')

dfs = full_bls(key = dict_api[user]
               , sector_list = sectors
               , df = df_peer_msa
               , dates = (year_start, year_end)
               , pre = survey
               , data_type = data_type)

ind_list = df_industries['industry_name'].values.tolist()

print('')
print("Reshaping pulled data")
print('')

for ii in range(len(dfs)):
    df_temp = dfs[ii]
    for col in df_temp.columns:
        df_temp[col] = df_temp[col].apply(lambda x: x*1000)
    df_temp = df_temp.reset_index(names = 'date')
    df_temp = pd.melt(df_temp
               , id_vars = 'date'
               , var_name = 'MSA'
               , value_name = ind_list[ii])
    dfs[ii] = df_temp


# Merge every single dataframe we will have together (should maybe includes a `how = 'left'`?)
# Each "Jobs" column should be named by sector code description
# Roll up total jobs in each specific industry codes to the mapped sectors
# Organize two shapes of data frames - "Long" (melted) and "Wide" (dcasted)
# Clean date field

def merge_dfs(df1, df2):
    return df1.merge(df2, on=['date', 'MSA'])
df_joined = reduce(merge_dfs, dfs)
df_bls1 = df_joined.melt(id_vars=['date', 'MSA'], 
                    var_name='Industry', 
                    value_name='Value')
var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))
df_bls1['Variable'] = df_bls1['Industry'].map(var_map)
df_bls1 = df_bls1[['date', 'MSA', 'Industry', 'Variable', 'Value']]
df_bls1 = df_bls1.groupby(['date', 'MSA', 'Variable'])['Value'].sum().reset_index()
df_bls2 = (df_bls1.pivot_table(index=['date', 'MSA'], columns='Variable', values='Value')).reset_index()
df_bls1['date'] = df_bls1['date'].dt.year
df_bls2['date'] = df_bls2['date'].dt.year
df_bls1 = df_bls1.rename(columns = {'date':'Year'})
df_bls2 = df_bls2.rename(columns = {'date':'Year'})


print('')
print("Finished ^_^..V..")

In [ ]:


# Displaying for QC
display(df_bls1, df_bls2)

In [ ]:
# Checking to see if this matches our prior results, 

# For 2014-01-01, Sac Total Private should = 647700.0, Sac Gov = 225300.0
display(df_bls1.loc[df_bls1['MSA'] == 'Sacramento-Roseville-Folsom, CA Metro Area'])

In [ ]:
# Set output name for .xlsx files
name_output_long_xlsx = [indicator_name, ' ', geography, ' ', estimate, ' ', survey, ' Long.xlsx']
name_output_wide_xlsx = [indicator_name, ' ', geography, ' ', estimate, ' ', survey, ' Wide.xlsx']

name_output_long_xlsx = "".join(name_output_long_xlsx)
name_output_wide_xlsx = "".join(name_output_wide_xlsx)

# Set output name for .csv files
name_output_csv = [indicator_name, '_', geography, '_', estimate, '_', survey, '.csv']
name_output_csv = "".join(name_output_csv)

print(name_output_long_xlsx)
print(name_output_wide_xlsx)
print(name_output_csv)

In [ ]:
# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out, indicator_name)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
print('CSV files exported here: '   + path_out_csv )


# Export to csv
df_bls1.to_csv(os.path.join(path_out_csv, name_output_csv), index = False)

# Export to excel
if percentages == 'Yes':
    # Export long
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_long_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_long_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_bls1.to_excel(writer, index = False, sheet_name = 'MSA')
    # Export wide
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_wide_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_wide_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_bls2     .to_excel(writer, index = False, sheet_name = 'MSA Total')
        df_bls2_perc.to_excel(writer, index = False, sheet_name = 'MSA Perc' )
else:
    # Export long
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_long_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_long_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_bls1.to_excel(writer, index = False, sheet_name = 'MSA')
    # Export wide
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_wide_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_wide_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_bls2.to_excel(writer, index = False, sheet_name = 'MSA')


print('')
print("Successfully exported")

***

## **Code Graveyard**


***

In [ ]:
# # Series stored as a dictionary (unemployment rate by ethnicity)
# series_dict = {
#     'LNS14000003': 'White',
#     'LNS14000006': 'Black',
#     'LNS14000009': 'Hispanic'}

# # series_dict = {'SMU48124200500000001': 'Total Jobs'}

# # Start year and end year
# dates = ('2008', '2017')

In [ ]:
# # Specify json as content type to return
# headers = {'Content-type': 'application/json'}

# # Submit the list of series as data
# data = json.dumps({
#     "seriesid": list(series_dict.keys()),
#     "startyear": dates[0],
#     "endyear": dates[1]})

# # Post request for the data
# p = requests.post(
#     '{}{}'.format(url, key),
#     headers=headers,
#     data=data).json()['Results']['series']

In [ ]:
# # Date index from first series
# date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]

# # Empty dataframe to fill with values
# df = pd.DataFrame()

# # Build a pandas series from the API results, p
# for s in p:
#     df[series_dict[s['seriesID']]] = pd.Series(
#         index = pd.to_datetime(date_list),
#         data = [i['value'] for i in s['data']]
#         ).astype(float).iloc[::-1]

# # Show last 5 results
# df.tail()

In [ ]:
# # Series stored as a dictionary
# series_dict = {
#     # 'LNS12000000': 'Agricultural Total Employment'
#     'SMU48124202023800001': 'Agricultural Total Employment1'
#     , 'SMU06409002023800001': 'Agricultural Total Employment2'
# }

# # Start year and end year
# dates = ('2022', '2024')

# # Specify json as content type to return
# headers = {'Content-type': 'application/json'}

# # Submit the list of series as data
# data = json.dumps({
#     "seriesid" : list(series_dict.keys()),
#     "startyear": dates[0],
#     "endyear"  : dates[1]
# })

# # Post request for the data
# p = requests.post(
#     '{}{}'.format(url, key),
#     headers=headers,
#     data=data).json()['Results']['series']
# # Date index from first series
# date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]

# # Empty dataframe to fill with values
# df = pd.DataFrame()

# # Build a pandas series from the API results, p
# for s in p:
#     df[series_dict[s['seriesID']]] = pd.Series(
#         index = pd.to_datetime(date_list),
#         data = [i['value'] for i in s['data']]
#         ).astype(float).iloc[::-1]

# # Show last 5 results
# df.tail()


In [ ]:
# Reading in MSA inputs
# df_params = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'BLS_MSA')
# indicator_name = df_params['indicator_name'].values[0]
# year_start     = int(df_params['year_start'    ].values[0])
# year_end       = int(df_params['year_end'      ].values[0])


# df_peer_msa = df_params[['msa', 'msa_label']]

# df_states = pd.read_excel(os.path.join(path_config0, "Area Codes.xlsx"), sheet_name = 'MSAcodes', dtype = {'State FIPS': object})
# df_peer_msa = df_peer_msa.merge(df_states[['MSA_ID', 'State FIPS']], left_on = 'msa', right_on = 'MSA_ID', how = 'left')
# df_peer_msa = df_peer_msa.drop('MSA_ID', axis = 1)

# print(indicator_name)
# print(year_start    )
# print(year_end      )
# df_peer_msa.head()

In [ ]:
# df_industries = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
#                               , sheet_name = 'industry_codes'
#                               , dtype = {'industry_code': object})
# df_industries = df_industries[df_industries['Include'] == 'Yes']
# df_industries = df_industries[df_industries['Indicator Name'].str.contains(indicator_name)] # Jobs_2
# df_industries.head()

In [ ]:
# df_datatypes = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
#                              , sheet_name = 'datatype_codes'
#                               , dtype = {'data_type_code': object})
# df_datatypes = df_datatypes[df_datatypes['Include'] == 'Yes']
# df_datatypes = df_datatypes[df_datatypes['Indicator Name'].str.contains(indicator_name)]
# df_datatypes.head()

In [ ]:
# # Subset
# df_peer_msa_in = df_peer_msa[df_peer_msa['msa_label'].str.contains('Sac|Austin')]
# df_peer_msa_in

In [ ]:
# Testing Agg = Total Nonfarm - Total Priv
# sectors = ['00000000', '05000000', '08000000']

# Inputs
# sectors    = list(df_industries['industry_code'].values)
# year_start = int(df_params['year_start'].values[0])
# year_end   = int(df_params['year_end'  ].values[0])
# data_type  = df_datatypes['data_type_code'].values[0]

# print(sectors   )
# print(year_start)
# print(year_end  )
# print(data_type )

# dfs = full_bls(key = dict_api[user]
#                , sector_list = sectors
#                , df = df_peer_msa_in
#                , dates = (year_start, year_end)
#                , pre = "SMU" # defines the survey
#                , data_type = data_type)

In [ ]:
# dfs_test[0] = dfs_test[0].rename(columns = {'Total Jobs':'05000000'})
# dfs_test[1] = dfs_test[1].rename(columns = {'Total Jobs':'90000000'})



# df_final = dfs_test[0].merge(dfs_test[1], on = ['date', 'MSA'])


In [ ]:
# from functools import reduce

# def merge_dfs(df1, df2):
#     return df1.merge(df2, on=['date', 'MSA'])

# # Use reduce to merge the entire list of DataFrames
# df_final = reduce(merge_dfs, dfs_test)

In [ ]:
# DOING JOBS 2

# Reading in MSA inputs
# df_params = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'BLS_MSA')
# indicator_name = df_params['indicator_name'].values[0]
# year_start     = int(df_params['year_start'    ].values[0])
# year_end       = int(df_params['year_end'      ].values[0])


# df_peer_msa = df_params[['msa', 'msa_label']]

# df_states = pd.read_excel(os.path.join(path_config0, "Area Codes.xlsx"), sheet_name = 'MSAcodes', dtype = {'State FIPS': object})
# df_peer_msa = df_peer_msa.merge(df_states[['MSA_ID', 'State FIPS']], left_on = 'msa', right_on = 'MSA_ID', how = 'left')
# df_peer_msa = df_peer_msa.drop('MSA_ID', axis = 1)

# print(indicator_name)
# print(year_start    )
# print(year_end      )
# df_peer_msa.head()

# df_industries = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
#                               , sheet_name = 'industry_codes'
#                               , dtype = {'industry_code': object})
# df_industries = df_industries[df_industries['Include'] == 'Yes']
# df_industries = df_industries[df_industries['Indicator Name'].str.contains(indicator_name)] # Jobs_2

# df_datatypes = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
#                              , sheet_name = 'datatype_codes'
#                               , dtype = {'data_type_code': object})
# df_datatypes = df_datatypes[df_datatypes['Include'] == 'Yes']
# df_datatypes = df_datatypes[df_datatypes['Indicator Name'].str.contains(indicator_name)]

# # Inputs
# sectors    = list(df_industries['industry_code'].values)
# year_start = int(df_params['year_start'].values[0])
# year_end   = int(df_params['year_end'  ].values[0])
# #data_type  = df_datatypes['data_type_code'].values[0] # I get an error here, probs cuz in the sheet it's defined as Jobs_1. 
# data_type = '01'

# print(sectors   )
# print(year_start)
# print(year_end  )
# print(data_type )

In [ ]:
# Shortening this down so we don't pull as much
# sectors = sectors[5:8]

# # have called the api 2 times today
# # dfs = full_bls(key = dict_api[user]
# #                , sector_list = sectors
# #                , df = df_peer_msa_in
# #                , dates = (year_start, year_end)
# #                , pre = "SMU" # defines the survey
# #                , data_type = data_type)


# # Goal of below chunk is to have each dataframe named by the industry it focuses on

# # Should we use Variable or Industry Name?
# ind_list = df_industries['industry_name'].values.tolist() # Change this to industry name

# # Have to subset here as well, we test more
# ind_list = ind_list[5:8]

# df_jobs_2 = dfs.copy()

# for ii in range(len(df_jobs_2)):
#     df_temp = df_jobs_2[ii]
#     for col in df_temp.columns:
#         df_temp[col] = df_temp[col].apply(lambda x: x*1000)
#     df_temp = df_temp.reset_index(names = 'date')
#     df_temp = pd.melt(df_temp
#                , id_vars = 'date'
#                , var_name = 'MSA'
#                , value_name = ind_list[ii]) # This is all i changed 
#     df_jobs_2[ii] = df_temp

# # from functools import reduce

# def merge_dfs(df1, df2):
#     return df1.merge(df2, on=['date', 'MSA'])

# # Function joins all of the Total jobs together, each of these total jobs column should be named by sector

# df_joined = reduce(merge_dfs, df_jobs_2)

# df_melted = df_joined.melt(id_vars=['date', 'MSA'], 
#                     var_name='Industry', 
#                     value_name='Value')

# # var_map is so we convert the industry names to the matching variable names we have in the excel sheet

# var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))

# # mapping

# df_melted['Variable'] = df_melted['Industry'].map(var_map)

# # Industry is kept, but we won't be summing using it. 

# df_melted = df_melted[['date', 'MSA', 'Industry', 'Variable', 'Value']] #'Variable',

# # Summing matches on date, msa, and var

# df_melted = df_melted.groupby(['date', 'MSA', 'Variable'])['Value'].sum().reset_index()

# # Outputting the unmelted dataframe

# df_unmelted = (df_melted.pivot_table(index=['date', 'MSA'], columns='Variable', values='Value')).reset_index()

In [ ]:
# df_unmelted = (df_melted.pivot_table(index=['date', 'MSA'], columns='Variable', values='Value')).reset_index()

# display(df_unmelted)

In [ ]:
# df_joined = reduce(merge_dfs, df_jobs_2)

# df_melted = df_joined.melt(id_vars=['date', 'MSA'], 
#                     var_name='Industry', 
#                     value_name='Value')

# var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))

# df_melted['Variable'] = df_melted['Industry'].map(var_map)

# # df_melted = df_melted[['date', 'MSA', 'Industry', 'Variable', 'Value']] #'Variable',

# df_melted = df_melted.groupby(['date', 'MSA', 'Variable'])['Value'].sum().reset_index() # If we groupby, and don't use 4th & 5th line, it gives what we want

In [ ]:
# df_melted[(df_melted['date'] == '2023-12-01') & (df_melted['MSA'] == 'Austin-Round Rock-Georgetown, TX Metro Area')]

In [ ]:
# df_melted['Industry'].value_counts()

In [ ]:
# df_final

# Melt this dataframe, have the three columns in a single row. Have this represented by a categorical variable.

# In this example, for each date we will have three rows corresponding to the specific date

# Merge the variable mapping name onto industry_name

# So then add all of the corresponding industry_names together. For instance industry_names that fall in health will be added together by date/msa.

# Will need to use melting/decasting(this is an r function need to find the equivalent)

# See broadband_2tractacs in sharepoint https://sacog.sharepoint.com/sites/RegionalMonitoringandReporting/Shared%20Documents/Forms/AllItems.aspx?CT=1714573103380&OR=OWA%2DNT%2DMail&CID=73e9a33f%2Df757%2D7f7c%2D630e%2D591d095884dd&id=%2Fsites%2FRegionalMonitoringandReporting%2FShared%20Documents%2FData%2FVibrant%20and%20Inclusive%20Places%2FPeople%20and%20Community%2FBroadband%2FBroadband%5F2&viewid=8fba2a3f%2Dea8a%2D4663%2Da31f%2Dc1760cb6922d